# Data Rescue Kit

Data work often fails for ordinary reasons: the file is in a different folder, a number arrived as text, a date did not parse, or an API returned no records.

Use this notebook like a first-aid kit. Find the symptom that looks like yours. Every section follows the same four steps:

1. **Symptom** — what you notice
2. **Diagnose** — what to inspect
3. **Fix** — a small change
4. **Verify** — evidence that the change worked

Do not apply a repair until you understand what it changes.

**Understand → Predict → Run → Change → Use → Practice → Connect**

[View on GitHub](https://github.com/mathguy-uf/MAD2502/blob/main/applied_data/04_data_rescue_kit.ipynb) · [Run in Google Colab](https://colab.research.google.com/github/mathguy-uf/MAD2502/blob/main/applied_data/04_data_rescue_kit.ipynb) · [Launch Binder/JupyterLab](https://mybinder.org/v2/gh/mathguy-uf/MAD2502/main?urlpath=lab/tree/applied_data/04_data_rescue_kit.ipynb)

Run the setup first. Then you can jump to the problem that matches yours.

In [ ]:
# If an import fails, uncomment and run this once, then restart the kernel.
# %pip install pandas

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
from IPython.display import display

## 1. `FileNotFoundError` or the wrong working directory

**Symptom:** `pd.read_csv(...)` raises `FileNotFoundError`, even though you believe the file exists.

**Diagnose:** print `Path.cwd()` to see the current working directory. Then inspect the path you are trying to use with `.exists()`. Relative paths begin from the current working directory, not necessarily from the folder you are looking at in the notebook interface.

**Fix:** correct the folder, filename, spelling, or capitalization.

**Verify:** check `.exists()` again and confirm that `pd.read_csv(...)` produces the expected rows.

**Predict:** which path below will exist?

In [ ]:
print("Current working directory:", Path.cwd())

with TemporaryDirectory() as folder:
    data_folder = Path(folder)
    correct_path = data_folder / "example.csv"
    correct_path.write_text(
        "item,value\nA,3\nB,5\n",
        encoding="utf-8",
    )

    wrong_path = data_folder / "wrong_folder" / "example.csv"

    print("Wrong path exists:", wrong_path.exists())
    print("Correct path exists:", correct_path.exists())

    try:
        pd.read_csv(wrong_path)
    except FileNotFoundError:
        print("Expected FileNotFoundError: that path does not exist.")

    fixed = pd.read_csv(correct_path)
    display(fixed)

    print("Verified row count:", len(fixed))

## 2. Numbers stored as strings, including `$` and commas

**Symptom:** a column that looks numeric has dtype `object` or `string`, or a calculation fails.

**Diagnose:** inspect `df.dtypes` and a few original values. Symbols such as `$` and commas make entries such as `"$1,200"` text.

**Fix:** remove the non-numeric characters, then use `pd.to_numeric(...)`.

**Verify:** inspect the new dtype and check which original values became missing.

**Predict:** which non-missing value below cannot be converted?

In [ ]:
money = pd.DataFrame({"raw": ["$1,200", "$0", "unknown", None]})

print("Before:")
print(money.dtypes)
display(money)

cleaned_text = money["raw"].astype("string")
cleaned_text = cleaned_text.str.replace("$", "", regex=False)
cleaned_text = cleaned_text.str.replace(",", "", regex=False)

money["value"] = pd.to_numeric(
    cleaned_text,
    errors="coerce",
)

print("\nAfter:")
print(money.dtypes)
display(money)

failed_conversion = money[money["raw"].notna() & money["value"].isna()]
print("Values that did not convert:")
display(failed_conversion)

## 3. Missing values (`NaN`)

**Symptom:** you see `NaN`, `None`, or blank cells, or a summary uses fewer rows than expected.

**Diagnose:** use `.isna().sum()` to count missing values and `.count()` to count known values.

**Fix:** choose a response that matches the meaning of the data. You might keep the missing value, remove a row for one specific calculation, or fill it only when you have a defensible replacement. Missing does **not** automatically mean zero.

**Verify:** report how many values were used in the calculation.

**Predict:** how many known values will the mean use below?

In [ ]:
scores = pd.DataFrame({"value": [4, None, 8]})

print("Missing values:", scores["value"].isna().sum())
print("Known values:", scores["value"].count())

mean_value = scores["value"].mean()

print("Mean of known values:", mean_value)
print(
    "Verified denominator:",
    scores["value"].count(),
    "of",
    len(scores),
)

## 4. Dates stored as text

**Symptom:** dates have dtype `object` or `string`, so sorting or date calculations behave like text.

**Diagnose:** inspect the raw values and determine their documented format.

**Fix:** use `pd.to_datetime(...)`. Give an explicit format when you know it. Here we deliberately support ISO dates and U.S. month/day/year dates.

**Verify:** inspect the parsed values and list anything that became `NaT` (missing date/time).

**Predict:** which entry will remain unparsed?

In [ ]:
dates = pd.Series(
    ["2026-10-01", "10/05/2026", "bad date"],
    name="raw",
)

iso_dates = pd.to_datetime(
    dates,
    format="%Y-%m-%d",
    errors="coerce",
)
us_dates = pd.to_datetime(
    dates,
    format="%m/%d/%Y",
    errors="coerce",
)

parsed_dates = iso_dates.fillna(us_dates)

date_check = pd.DataFrame(
    {
        "raw": dates,
        "parsed": parsed_dates,
    }
)
display(date_check)

print(
    "Unparsed values:",
    dates[parsed_dates.isna()].tolist(),
)

## 5. Duplicate rows

**Symptom:** the same record appears more than once, or row counts are larger than expected.

**Diagnose:** use `.duplicated(...)` to inspect suspected duplicates. Decide what identifies one real record. Equal values are not automatically duplicates.

**Fix:** remove duplicates only when you have evidence that they represent the same record. In this example, repeated `event_id` means the same imported event.

**Verify:** compare row counts before and after.

**Predict:** how many rows should remain?

In [ ]:
events = pd.DataFrame(
    {
        "event_id": [1, 2, 2],
        "attendance": [10, 20, 20],
    }
)

suspected = events[
    events.duplicated(
        subset=["event_id"],
        keep=False,
    )
]
print("Suspected duplicates:")
display(suspected)

unique_events = events.drop_duplicates(subset=["event_id"])

print(
    "Rows before / after:",
    len(events),
    len(unique_events),
)

## 6. Strange column names or `KeyError`

**Symptom:** `df["cost"]` raises `KeyError`, even though a column looks like it says cost.

**Diagnose:** print `list(df.columns)`. Hidden spaces and capitalization are part of the column name.

**Fix:** clean names carefully with `strip()`, lowercase, and underscores.

**Verify:** print the new names and make sure they are unique.

**Predict:** what will the two cleaned names be?

In [ ]:
table = pd.DataFrame(
    {
        " Event Name ": ["A"],
        " Cost ": [10],
    }
)

print("Before:", list(table.columns))

table.columns = table.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)

print("After:", list(table.columns))
assert table.columns.is_unique

display(table[["event_name", "cost"]])

## 7. Empty DataFrame after filtering

**Symptom:** a filter produces a DataFrame with zero rows.

**Diagnose:** check `.empty`, `.shape`, the filter condition, and the actual range of the relevant column.

**Fix:** correct the condition only if the condition was wrong. Do not invent rows just because the result is empty.

**Verify:** rerun the filter and check the resulting row count.

**Predict:** will `cost > 5000` return any rows here?

In [ ]:
values = pd.DataFrame({"cost": [0, 10, 20]})

selected = values[values["cost"] > 5000]

print("Selected shape:", selected.shape)
print("Selected is empty:", selected.empty)
print(
    "Actual cost range:",
    values["cost"].min(),
    "to",
    values["cost"].max(),
)

corrected = values[values["cost"] >= 10]
print("Rows after corrected example filter:", len(corrected))
display(corrected)

## 8. Empty API response

**Symptom:** an API request succeeds, but the returned JSON contains an empty list: `[]`.

**Diagnose:** first check the HTTP status code in your live request. If it is successful, then check the query parameters, region, species code, date range, and other filters. A successful request can legitimately have no matching records.

**Fix:** correct the query only when you find a mistake. Otherwise, an empty result may be the correct answer.

**Verify:** check `len(records)` before building summaries. If you create a DataFrame, preserve the expected column names so later code can still inspect its structure.

**Predict:** what shape will the DataFrame have below?

In [ ]:
# This represents JSON returned by a successful API request.
status_code = 200
records = []

print("Status code:", status_code)
print("Number of records:", len(records))

api_data = pd.DataFrame(
    records,
    columns=["speciesCode", "howMany"],
)

print("Shape:", api_data.shape)
print("Empty:", api_data.empty)

if api_data.empty:
    print("No matching records. Check the query before computing a summary.")

## 9. Nested JSON

**Symptom:** API data contains dictionaries inside other dictionaries, so the fields you want are not simple columns yet.

**Diagnose:** inspect `type(...)`, dictionary keys, and one small record before converting everything.

**Fix:** select the list of records, then use `pd.json_normalize(...)` to flatten nested dictionaries.

**Verify:** inspect the resulting column names and first rows.

**Predict:** which dictionary key contains the list of records?

In [ ]:
payload = {
    "results": [
        {
            "name": "A",
            "location": {"city": "Gainesville"},
        }
    ]
}

print("Payload type:", type(payload))
print("Payload keys:", list(payload.keys()))

records = payload["results"]

print("Records type:", type(records))
print("First record:", records[0])

flat = pd.json_normalize(records)

print("Flattened columns:", list(flat.columns))
display(flat)

## 10. Wrong Python object type

**Symptom:** an operation gives an error such as `AttributeError: 'list' object has no attribute 'head'`.

**Diagnose:** use `type(...)`. Different Python objects support different operations. A list uses positions, a dictionary uses keys, and a DataFrame has rows and named columns.

**Fix:** use the operation that belongs to the object you actually have, or convert the data when a DataFrame is what you need.

**Verify:** inspect the new type and successfully run the intended operation.

**Predict:** why will `records.head()` fail below?

In [ ]:
records = [
    {"item": "A", "value": 3},
    {"item": "B", "value": 5},
]

print("Before:", type(records))

try:
    records.head()
except AttributeError as error:
    print("Expected error:", error)

table = pd.DataFrame(records)

print("After:", type(table))
display(table.head())

## Practice

Pick any two sections above.

1. **Understand:** explain the problem in one sentence.
2. **Predict:** change one input and predict the result.
3. **Run:** run the original example.
4. **Change:** make your one-input change.
5. **Use:** apply the same diagnosis to one of your own project variables or files.
6. **Practice:** write down the symptom, diagnosis, fix, and verification.
7. **Connect:** explain how this check could prevent a wrong conclusion in a larger data project.

## A good rescue habit

Before trusting a cleaned dataset, record:

- what one row represents,
- the number of rows before and after cleaning,
- missing-value counts,
- column names and dtypes,
- any values that failed conversion,
- and why each repair was appropriate.

Then restart the kernel and run the notebook from the top. A repair is not complete until you can verify it.